In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_54.pickle

In [ ]:
Out.clear()

In [ ]:
%%RecordEvent
%%time
### cell 55 ###

def grab_subset_of_data_67(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_67(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_67(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_67(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_67(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_67(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_67(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_67(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_67(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_67(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_67(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_67(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_67(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_67(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_67(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_67(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
question_of_interest_cell67 = 'Which of the following cloud computing platforms do you use? (Select all that apply)'
question_of_interest_old_cell67 = 'Which of the following cloud computing platforms do you use on a regular basis?'
question_of_interest_old_2 = 'Which of the following cloud computing services have you used at work or school in the last 5 years?'
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace('Amazon Web Services (AWS)', 'Amazon Web Services (AWS) ', regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace('Google Cloud Platform (GCP)', 'Google Cloud Platform (GCP) ', regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace('Microsoft Azure', 'Microsoft Azure ', regex=False)
responses_df_2021.columns = responses_df_2021.columns.str.replace(question_of_interest_old_cell67, question_of_interest_cell67, regex=False)
responses_df_2020.columns = responses_df_2020.columns.str.replace(question_of_interest_old_cell67, question_of_interest_cell67, regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(question_of_interest_old_cell67, question_of_interest_cell67, regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(question_of_interest_old_2, question_of_interest_cell67, regex=False)
(cloud_df_combined, cloud_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(question_of_interest_cell67)
cloud_df_combined_percentages = convert_df_of_counts_to_percentages_67(cloud_df_combined, cloud_df_combined_counts)
cloud_df_combined_percentages_cell67 = cloud_df_combined_percentages.loc[:, ['year', 'Amazon Web Services (AWS) ', 'Google Cloud Platform (GCP) ', 'Microsoft Azure ']]
df_cell67 = cloud_df_combined_percentages_cell67.melt(id_vars=['year'], value_vars=['Amazon Web Services (AWS) ', 'Google Cloud Platform (GCP) ', 'Microsoft Azure '])
df_cell67 = df.rename(columns={'variable': ''})
df_cell67.info()

In [ ]:
%%RecordEvent
orig_output = Out.get(5)

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_55.pickle